# สัปดาห์ที่ 8: OOP Fundamentals & Refactoring the CLI Task Manager
### แบบฝึกหัดปฏิบัติการเขียนโปรแกรมเชิงวัตถุ (Object-Oriented Programming) ด้วยภาษา Python

สมุดงานนี้ออกแบบสำหรับนักศึกษาชั้นปีที่ 2 เพื่อฝึกฝนการเปลี่ยนกระบวนทัศน์จาก **Procedural Programming** ในสัปดาห์ที่ 7 มาเป็น **Object-Oriented Programming (OOP)** ในสัปดาห์ที่ 8

---
### 🎯 วัตถุประสงค์การเรียนรู้:
1. กำหนด Class และสร้าง Instance (Object)
2. กำหนด Constructor (`__init__`) และเข้าใจการทำงานของ `self`
3. สร้าง Instance Methods และ Instance Attributes
4. ใช้งาน Dunder Methods: `__str__` (User format) และ `__repr__` (Debug format)
5. แปลง Object เป็น Dictionary (`to_dict()`) เพื่อบันทึกเป็น JSON (Serialization / Deserialization)

## ขั้นตอนที่ 0: สร้างโฟลเดอร์สำหรับโปรเจกต์

In [ ]:
!mkdir -p data
!mkdir -p src
!mkdir -p tests
print("Directories created successfully!")

## ขั้นตอนที่ 1: สร้างโมดูล `src/task.py` (คลาส Task)
คลาส `Task` ทำหน้าที่เป็นตัวแทนข้อมูลของงานแต่ละรายการ

In [ ]:
%%writefile src/task.py
# src/task.py

class Task:
    """
    Represents a single task in the task manager.
    """
    def __init__(self, id: int, description: str, completed: bool = False):
        self.id = id
        self.description = description
        self.completed = completed

    def mark_complete(self):
        """Marks the task as completed."""
        self.completed = True

    def to_dict(self):
        """Converts Task object to a dictionary for JSON serialization."""
        return {
            "id": self.id,
            "description": self.description,
            "completed": self.completed
        }

    def __str__(self):
        """User-friendly string representation."""
        status = "Completed" if self.completed else "Pending"
        return f"ID: {self.id} | Description: {self.description} | Status: {status}"

    def __repr__(self):
        """Developer-friendly string representation."""
        return f"Task(id={self.id}, description='{self.description}', completed={self.completed})"


## ขั้นตอนที่ 2: สร้างโมดูล `src/task_manager.py` (คลาส TaskManager)
คลาสนี้จะทำหน้าที่เป็น Controller จัดการคอลเลกชันของ Task Objects และจัดการอ่าน/เขียนไฟล์ JSON

In [ ]:
%%writefile src/task_manager.py
# src/task_manager.py
import json
import os
from .task import Task

class TaskManager:
    """
    Manages the collection of Task objects, including loading, saving,
    and performing operations like add, list, complete, and delete.
    """
    def __init__(self, data_file='data/tasks.json'):
        self.data_file = data_file
        self._data_file_path = os.path.join(os.getcwd(), self.data_file)
        self.tasks = self._load_tasks()
        self.next_id = self._get_next_task_id()

    def _get_next_task_id(self):
        if not self.tasks:
            return 1
        return max(task.id for task in self.tasks) + 1

    def _load_tasks(self):
        if not os.path.exists(self._data_file_path):
            os.makedirs(os.path.dirname(self._data_file_path), exist_ok=True)
            return []
        try:
            with open(self._data_file_path, 'r', encoding='utf-8') as f:
                raw_tasks = json.load(f)
                return [Task(t['id'], t['description'], t['completed']) for t in raw_tasks]
        except (json.JSONDecodeError, FileNotFoundError):
            print("Warning: tasks.json is empty or corrupted. Starting with an empty task list.")
            return []

    def _save_tasks(self):
        tasks_as_dicts = [task.to_dict() for task in self.tasks]
        os.makedirs(os.path.dirname(self._data_file_path), exist_ok=True)
        with open(self._data_file_path, 'w', encoding='utf-8') as f:
            json.dump(tasks_as_dicts, f, indent=4, ensure_ascii=False)

    def add_task(self, description):
        new_task = Task(self.next_id, description)
        self.tasks.append(new_task)
        self.next_id += 1
        self._save_tasks()
        print(f"Task '{description}' added with ID {new_task.id}.")
        return new_task

    def list_tasks(self):
        if not self.tasks:
            print("No tasks found.")
            return
        print("\n--- Your Tasks ---")
        for task in self.tasks:
            print(task)
        print("------------------")

    def complete_task(self, task_id):
        found = False
        for task in self.tasks:
            if task.id == task_id:
                if task.completed:
                    print(f"Task ID {task_id} is already completed.")
                else:
                    task.mark_complete()
                    self._save_tasks()
                    print(f"Task ID {task_id} marked as completed.")
                found = True
                break
        if not found:
            print(f"Error: Task with ID {task_id} not found.")
        return found

    def delete_task(self, task_id):
        original_len = len(self.tasks)
        self.tasks = [task for task in self.tasks if task.id != task_id]
        if len(self.tasks) < original_len:
            self._save_tasks()
            print(f"Task ID {task_id} deleted successfully.")
            return True
        else:
            print(f"Error: Task with ID {task_id} not found.")
            return False


## ขั้นตอนที่ 3: สร้างไฟล์แพ็กเกจ `src/__init__.py`

In [ ]:
%%writefile src/__init__.py
# src/__init__.py


## ขั้นตอนที่ 4: รันโปรแกรม CLI ใน Google Colab

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.getcwd(), 'src'))
from task_manager import TaskManager

def display_menu():
    print("\n--- OOP Task Manager Menu ---")
    print("1. Add Task")
    print("2. List Tasks")
    print("3. Complete Task")
    print("4. Delete Task")
    print("5. Exit")
    print("-------------------------")

def main_colab():
    manager = TaskManager()
    while True:
        display_menu()
        choice = input("Enter your choice (1-5): ").strip()
        if choice == '1':
            description = input("Enter task description: ").strip()
            if description:
                manager.add_task(description)
            else:
                print("Task description cannot be empty.")
        elif choice == '2':
            manager.list_tasks()
        elif choice == '3':
            task_id_str = input("Enter ID of task to complete: ").strip()
            try:
                task_id = int(task_id_str)
                manager.complete_task(task_id)
            except ValueError:
                print("Invalid input. Please enter a number for Task ID.")
        elif choice == '4':
            task_id_str = input("Enter ID of task to delete: ").strip()
            try:
                task_id = int(task_id_str)
                manager.delete_task(task_id)
            except ValueError:
                print("Invalid input. Please enter a number for Task ID.")
        elif choice == '5':
            print("Exiting Task Manager. Goodbye!")
            break
        else:
            print("Invalid choice. Please enter a number between 1 and 5.")

main_colab()
